# Crop Price Dataset Cleaning & Preparation
## Advanced Data Cleaning Pipeline using Pandas & NumPy

---

### **Project Structure Overview**
* **Section 0: Project Setup & Baseline Data Inspection**
  * 0.1 Library Imports
  * 0.2 Dataset Loading
  * 0.3 Dimensions & Data Preview
  * 0.4 Column Names & Data Types
  * 0.5 Dataset Summary (`info()`)
  * 0.6 Initial Missing Values & Duplicate Scan
* **Section 1: Problem 1 — Missing Values Analysis & Imputation**
  * 1.1 Missing Value Discovery & Statistical Summary (Steps 1–6)
  * 1.2 Weather Features Imputation: Domain-Specific Hierarchical Logic (Steps 7–9)
  * 1.3 Time-Series Pre-processing & Sorting (Step 10 Prep)
  * 1.4 Supply / Quantity Imputation: Grouped Time-Series Interpolation (Steps 10–11)
  * 1.5 Price Imputation: Grouped Time-Series Interpolation (Steps 12–15)
  * 1.6 Quality Assurance & Intermediate Checkpoint (Steps 16–18)

---

# SECTION 0: Project Setup & Baseline Data Inspection
In this initial phase, we set up our Python runtime environment, import necessary scientific computing libraries, load the raw (dirty) dataset, and inspect its structure, dimensions, column schema, and data types.

### Sub-section 0.1: Import Required Libraries
We import `pandas` for data manipulation and tabular transformations, and `numpy` for numerical operations and handling NaN representations.

In [11]:
# Import core data analysis libraries
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


### Sub-section 0.2: Load the Dirty Dataset
We load the uncleaned raw CSV file into a pandas DataFrame. We specify `low_memory=False` to ensure smooth parsing across all column data types.

In [12]:
# Define the path to the raw/dirty CSV file
file_path = "nagpur_crop_price_DIRTY_FINAL_2022_2026.csv"

# Load dataset
df = pd.read_csv(file_path, low_memory=False)

print("Dataset loaded successfully!")

Dataset loaded successfully!


### Sub-section 0.3: Basic Dimensions and First Look
Inspect the total number of records (rows) and attributes (columns) present, followed by viewing the first 5 rows.

In [13]:
# Display dimensions
print(f"Number of rows    : {df.shape[0]:,}")
print(f"Number of columns : {df.shape[1]}")

# Display first 5 rows
display(df.head())

Number of rows    : 282,504
Number of columns : 21


,date,state,district,market,commodity,arrival_quantity,min_price,max_price,modal_price,msp,...,export_quantity,import_value,export_value,fuel_cost,temperature,humidity,msp_applicable,serial_no,old_index,data_entry_user
0,01-01-2022,Maharastra,Wardha,HiNgNa mArKeT,COTTON CROP,"1,244 units",6248,6867,NaN,5726,...,2865,193.33,213.32,-30,150°C,-20.0,2,1,0,collector_01
1,01-01-2022,Maharashtra,Nagpur,Kalmeshwar,Cotton,1078,6282/-,6808,6560,5726,...,3183,168.96,223.17,93.43,20.1,NaN,1,2,1,collector_02
2,01-01-2022,Maharashtra,Nagpur,Kamthi,COTTON,1387,6232,69.53 per kg,6625,5726,...,2049,293.68,140.92,93.55,20.2,58.0,1,3,2,collector_03
3,01-01-2022,Maharashtra,Nagpur,Katol,Cotton,1399,63320,7024,6650,5726,...,2794,198.52,193.76,93.32,20.3,58.2,1,4,3,collector_04
4,01-Jan-22,Maharashtra,Nagpur,Mandhal,Cotton crop,875,6248,6933,6563,5726,...,3418,145.95,239.66,93.23,20.4,58.3,1,5,4,collector_05


### Sub-section 0.4: Check Column Names & Data Types
Verify the exact column headers and current inferred data types for each feature.

In [14]:
# Column names
print("Column Names:")
print(df.columns.tolist())

# Data types
print("\nData Types:")
print(df.dtypes)

Column Names:
['date', 'state', 'district', 'market', 'commodity', 'arrival_quantity', 'min_price', 'max_price', 'modal_price', 'msp', 'import_quantity', 'export_quantity', 'import_value', 'export_value', 'fuel_cost', 'temperature', 'humidity', 'msp_applicable', 'serial_no', 'old_index', 'data_entry_user']

Data Types:
date                 object
state                object
district             object
market               object
commodity            object
arrival_quantity     object
min_price            object
max_price            object
modal_price          object
msp                  object
import_quantity      object
export_quantity      object
import_value        float64
export_value        float64
fuel_cost            object
temperature          object
humidity            float64
msp_applicable        int64
serial_no             int64
old_index             int64
data_entry_user      object
dtype: object


### Sub-section 0.5: Comprehensive Dataset Information
Run `df.info()` to see memory usage, non-null counts, and data types at a glance.

In [15]:
print("Dataset Information:")
df.info()

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 282504 entries, 0 to 282503
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   date              282504 non-null  object 
 1   state             282504 non-null  object 
 2   district          282504 non-null  object 
 3   market            282504 non-null  object 
 4   commodity         282504 non-null  object 
 5   arrival_quantity  277821 non-null  object 
 6   min_price         278332 non-null  object 
 7   max_price         278304 non-null  object 
 8   modal_price       273092 non-null  object 
 9   msp               282504 non-null  object 
 10  import_quantity   277797 non-null  object 
 11  export_quantity   282504 non-null  object 
 12  import_value      282504 non-null  float64
 13  export_value      282504 non-null  float64
 14  fuel_cost         282504 non-null  object 
 15  temperature       277795 non-null  object 
 16 

### Sub-section 0.6: Initial Overview of Missing Values & Duplicates
Perform a quick baseline check to get an initial sense of missing values and duplicate rows in the raw dataset.

In [16]:
# Baseline missing values
print("Missing values per column:")
print(df.isnull().sum())

# Baseline duplicate rows
print(f"\nNumber of duplicate rows: {df.duplicated().sum():,}")

Missing values per column:
date                   0
state                  0
district               0
market                 0
commodity              0
arrival_quantity    4683
min_price           4172
max_price           4200
modal_price         9412
msp                    0
import_quantity     4707
export_quantity        0
import_value           0
export_value           0
fuel_cost              0
temperature         4709
humidity            4695
msp_applicable         0
serial_no              0
old_index              0
data_entry_user        0
dtype: int64

Number of duplicate rows: 12,000


---
# SECTION 1: Problem 1 — Missing Values Analysis & Imputation

### Strategy & Pipeline Architecture:
1. **Detection & Profiling**: Identify missing counts, percentages, and inspect actual affected rows.
2. **Weather Features (`temperature`, `humidity`)**:
   - Impute via **Date + Market Median** (since on the same date in the same market, weather is identical across crops).
   - *Fallback 1*: **Market Median**.
   - *Fallback 2*: **Overall Median**.
3. **Quantity & Price Features (`arrival_quantity`, `import_quantity`, `modal_price`, `min_price`, `max_price`)**:
   - Sort chronologically by `market`, `commodity`, and `date`.
   - Impute via **Grouped Linear Time-Series Interpolation** within each `(market, commodity)` series.
   - *Fallback 1*: **Market + Commodity Median**.
   - *Fallback 2*: **Overall Column Median**.
4. **Validation & Checkpoint**: Confirm 0 missing cells in target columns and export checkpoint CSV.

---

## Sub-section 1.1: Missing Value Discovery & Statistical Summary

#### Step 1: Detect Missing Values Across All Columns

#### Step 2: Isolate and Show Only Problematic Columns

In [17]:
# Detect missing values across all columns
missing_values = df.isnull().sum()

# Show only columns containing missing values
missing_columns = missing_values[missing_values > 0]

print("Columns containing missing values:")
print(missing_columns)

Columns containing missing values:
arrival_quantity    4683
min_price           4172
max_price           4200
modal_price         9412
import_quantity     4707
temperature         4709
humidity            4695
dtype: int64


#### Step 3: Calculate Total Missing Cells

In [18]:
# Total count of all missing cells across the dataset
total_missing = df.isnull().sum().sum()

print(f"Total missing cells: {total_missing:,}")

Total missing cells: 36,578


#### Step 4: Calculate Missing Percentage Summary Table

In [19]:
# Calculate missing percentage for every column
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage (%)": missing_percentage.round(2)
})

# Show only columns with missing values sorted descending
missing_summary = missing_summary[missing_summary["Missing Count"] > 0].sort_values(
    by="Missing Count", ascending=False
)

print("Missing Value Summary Table:")
display(missing_summary)

Missing Value Summary Table:


,Missing Count,Missing Percentage (%)
modal_price,9412,3.33
temperature,4709,1.67
import_quantity,4707,1.67
humidity,4695,1.66
arrival_quantity,4683,1.66
max_price,4200,1.49
min_price,4172,1.48


#### Step 5: Inspect Rows Containing Missing Values
Before applying automated imputation, inspect actual dirty rows to observe how missing values manifest across records.

In [20]:
# Display sample rows containing at least one missing value
rows_with_missing = df[df.isnull().any(axis=1)]

print(f"Number of rows containing at least one missing value: {len(rows_with_missing):,}")
print("Sample dirty rows (First 10 records):")
display(rows_with_missing.head(10))

Number of rows containing at least one missing value: 34,628
Sample dirty rows (First 10 records):


,date,state,district,market,commodity,arrival_quantity,min_price,max_price,modal_price,msp,...,export_quantity,import_value,export_value,fuel_cost,temperature,humidity,msp_applicable,serial_no,old_index,data_entry_user
0,01-01-2022,Maharastra,Wardha,HiNgNa mArKeT,COTTON CROP,"1,244 units",6248,6867,NaN,5726,...,2865,193.33,213.32,-30,150°C,-20.0,2,1,0,collector_01
1,01-01-2022,Maharashtra,Nagpur,Kalmeshwar,Cotton,1078,6282/-,6808,6560,5726,...,3183,168.96,223.17,93.43,20.1,NaN,1,2,1,collector_02
9,01-01-2022,Maharashtra,Nagpur,RAMTEK,Cotton,979,6283,7008,6602,5726,...,2965,176.68,203.85,93.38,NaN,59.0,1,10,9,collector_02
16,"January 01, 2022",Maharashtra,Nagpur,Katol,Rice,2146,4699,5212,NaN,1940,...,3671,185.67,125.87,93.67,20.3,58.2,1,17,16,collector_01
18,01-01-2022,Maharashtra,Bhandara,MAUDA,RiCe cRoP,1649,4609,51.30 per kg,4830,1940,...,3285,160.15,108.77,93.57,NaN,NaN,1,19,18,collector_03
23,01-01-2022,Maharashtra,Nagpur,Ramtek,Rice Crop,NaN,"4,678",5139,4886,1940,...,3513,162.15,118.89,93.39,150°C,59.0,1,23,22,collector_07
24,01-01-2022,Maharashtra,Nagpur,Savner,Rice,1874,4768,5280,4995,1940,...,3073,188.79,99.94,93.42,21.2,59.2,1,24,23,collector_08
28,01-01-2022,Maharashtra,Nagpur,Kalmeshwar,Wheat,992,2494,2779,2617,1975,...,1645,72.13,40.48,93.51,NaN,57.9,1,28,27,collector_04
32,01-01-2022,Maharashtra,Nagpur,Mauda,Wheat,922,NaN,2811,2631,1975,...,1304,84.74,34.73,93.44,20.5,58.4,1,32,31,collector_08
33,01-01-2022,Maharashtra,Nagpur,Nagpur,wheat,1168,2550,2817,NaN,1975,...,1268,104.32,33.13,93.57,20.7,58.6,1,33,32,collector_01


#### Step 6: Identify Target Columns for Cleaning

In [21]:
# Define the explicit list of columns to be cleaned
missing_value_columns = [
    "temperature",
    "humidity",
    "arrival_quantity",
    "import_quantity",
    "min_price",
    "max_price",
    "modal_price"
]

print("Columns targeted for missing value imputation:")
print(missing_value_columns)

Columns targeted for missing value imputation:
['temperature', 'humidity', 'arrival_quantity', 'import_quantity', 'min_price', 'max_price', 'modal_price']


## Sub-section 1.2: Weather Features Imputation (Domain-Specific Hierarchical Logic)

> **Domain Rationale:**
> Weather conditions (temperature and humidity) depend on location and calendar day, not on the specific crop being traded. Multiple commodities recorded in the same market on the same date share the same local climate.
>
> **Hierarchy:**
> 1. Grouped median by `(date, market)`
> 2. Grouped median by `market` (Fallback 1)
> 3. Global median across entire dataset (Fallback 2)

#### Step 7: Handle Missing Temperature (Date + Market Imputation)

In [22]:
# Convert temperature to numeric
df["temperature"] = pd.to_numeric(df["temperature"], errors="coerce")

In [23]:
# Calculate median temperature for each (date + market) combination
temperature_median = (
    df.groupby(["date", "market"])["temperature"]
      .transform("median")
)

# Impute using date + market median
df["temperature"] = df["temperature"].fillna(temperature_median)

print(f"Remaining missing temperature after Step 7: {df['temperature'].isnull().sum():,}")

Remaining missing temperature after Step 7: 576


#### Step 8: Temperature Fallbacks (Market Median $\rightarrow$ Overall Median)

In [24]:
# Fallback 1: Market-wise median
market_temperature_median = (
    df.groupby("market")["temperature"]
      .transform("median")
)
df["temperature"] = df["temperature"].fillna(market_temperature_median)

# Fallback 2: Overall global median
df["temperature"] = df["temperature"].fillna(df["temperature"].median())

print(f"Remaining missing temperature after fallbacks: {df['temperature'].isnull().sum():,}")

Remaining missing temperature after fallbacks: 0


#### Step 9: Handle Missing Humidity (Date + Market $\rightarrow$ Market $\rightarrow$ Overall)

In [25]:
# Primary Imputation: Date + Market median
humidity_median = (
    df.groupby(["date", "market"])["humidity"]
      .transform("median")
)
df["humidity"] = df["humidity"].fillna(humidity_median)

# Fallback 1: Market-wise median
market_humidity_median = (
    df.groupby("market")["humidity"]
      .transform("median")
)
df["humidity"] = df["humidity"].fillna(market_humidity_median)

# Fallback 2: Overall global median
df["humidity"] = df["humidity"].fillna(df["humidity"].median())

print(f"Remaining missing humidity after all steps: {df['humidity'].isnull().sum():,}")

Remaining missing humidity after all steps: 0


## Sub-section 1.3: Time-Series Sorting & Data Preparation

> **Preparation for Interpolation:**
> Market arrivals, import quantities, and prices evolve as time-series per `(market, commodity)`.
> To perform accurate linear interpolation, we must ensure dates are parsed into `datetime` and records are sorted chronologically within each market and commodity group.

In [26]:
# Convert date column with mixed formats to datetime
df["date"] = pd.to_datetime(df["date"], format="mixed", dayfirst=True)

# Verify missing values in date column
print(f"Missing values in date: {df['date'].isnull().sum()}")

# Sort data chronologically within each market and commodity pair
df = df.sort_values(["market", "commodity", "date"]).reset_index(drop=True)

print("Dataset successfully sorted by [market, commodity, date]!")
display(df[["date", "market", "commodity", "modal_price", "arrival_quantity"]].head())


Missing values in date: 0
Dataset successfully sorted by [market, commodity, date]!


,date,market,commodity,modal_price,arrival_quantity
0,2025-01-14,BHIWAPUR,Green Chilli,2786,800
1,2023-02-28,BHIWAPUR,Onion,3906,2046
2,2023-12-01,BHIWAPUR,Orange,8762,1034
3,2024-02-26,BHIWAPUR,Orange,13470,761
4,2024-07-23,BHIWAPUR,PeA,5319,411


## Sub-section 1.4: Supply / Quantity Features Imputation (Grouped Time-Series Interpolation)

> **Approach:**
> Linear interpolation along the timeline of each `(market, commodity)` group captures local seasonal and weekly trends. Any boundary NaNs are resolved using the group median followed by overall median.

#### Step 10: Handle Missing Arrival Quantity

In [27]:
# Convert arrival_quantity to numeric
df["arrival_quantity"] = pd.to_numeric(
    df["arrival_quantity"],
    errors="coerce"
)

In [28]:
# Step 10: Linear interpolation within each (market, commodity) group
df["arrival_quantity"] = (
    df.groupby(["market", "commodity"])["arrival_quantity"]
      .transform(lambda x: x.interpolate(method="linear"))
)

# Fallback 1: Market + commodity median
arrival_median = (
    df.groupby(["market", "commodity"])["arrival_quantity"]
      .transform("median")
)
df["arrival_quantity"] = df["arrival_quantity"].fillna(arrival_median)

# Fallback 2: Overall global median
df["arrival_quantity"] = df["arrival_quantity"].fillna(df["arrival_quantity"].median())

print(f"Remaining missing arrival_quantity: {df['arrival_quantity'].isnull().sum():,}")

Remaining missing arrival_quantity: 0


#### Step 11: Handle Missing Import Quantity

In [29]:
# Step 11: Handle Missing Import Quantity

# Convert import_quantity to numeric
df["import_quantity"] = pd.to_numeric(
    df["import_quantity"],
    errors="coerce"
)

# Linear interpolation within each (market, commodity) group
df["import_quantity"] = (
    df.groupby(["market", "commodity"])["import_quantity"]
      .transform(lambda x: x.interpolate(method="linear"))
)

# Fallback 1: Market + commodity median
import_median = (
    df.groupby(["market", "commodity"])["import_quantity"]
      .transform("median")
)

df["import_quantity"] = df["import_quantity"].fillna(import_median)

# Fallback 2: Overall global median
df["import_quantity"] = df["import_quantity"].fillna(
    df["import_quantity"].median()
)

print(
    f"Remaining missing import_quantity: "
    f"{df['import_quantity'].isnull().sum():,}"
)

Remaining missing import_quantity: 0


## Sub-section 1.5: Price Features Imputation (Grouped Time-Series Interpolation)

> **Domain Rationale:**
> Crop prices (`modal_price`, `min_price`, `max_price`) depend heavily on specific crop economics, market geography, and seasonal cycles.
> We use linear time-series interpolation per `(market, commodity)` group to reconstruct historical price movements without distorting commodity price levels.

#### Step 12 & 13: Handle Missing Modal Price (`modal_price`)

In [30]:
# Step 12 & 13: Handle Missing Modal Price

# Convert modal_price to numeric
df["modal_price"] = pd.to_numeric(
    df["modal_price"],
    errors="coerce"
)

# Linear interpolation within each (market, commodity) group
df["modal_price"] = (
    df.groupby(["market", "commodity"])["modal_price"]
      .transform(lambda x: x.interpolate(method="linear"))
)

# Fallback 1: Market + commodity median
modal_median = (
    df.groupby(["market", "commodity"])["modal_price"]
      .transform("median")
)

df["modal_price"] = df["modal_price"].fillna(modal_median)

# Fallback 2: Overall global median
df["modal_price"] = df["modal_price"].fillna(
    df["modal_price"].median()
)

print(
    f"Remaining missing modal_price: "
    f"{df['modal_price'].isnull().sum():,}"
)

Remaining missing modal_price: 0


#### Step 14: Handle Missing Minimum Price (`min_price`)

In [31]:
# Step 14: Handle Missing Min Price

# Convert min_price to numeric
df["min_price"] = pd.to_numeric(
    df["min_price"],
    errors="coerce"
)

# Linear interpolation within each (market, commodity) group
df["min_price"] = (
    df.groupby(["market", "commodity"])["min_price"]
      .transform(lambda x: x.interpolate(method="linear"))
)

# Fallback 1: Market + commodity median
min_price_median = (
    df.groupby(["market", "commodity"])["min_price"]
      .transform("median")
)

df["min_price"] = df["min_price"].fillna(min_price_median)

# Fallback 2: Overall global median
df["min_price"] = df["min_price"].fillna(
    df["min_price"].median()
)

print(
    f"Remaining missing min_price: "
    f"{df['min_price'].isnull().sum():,}"
)

Remaining missing min_price: 0


In [32]:
# Convert all numerical columns from object/string to numeric

numeric_cols = [
    "temperature",
    "arrival_quantity",
    "import_quantity",
    "modal_price",
    "min_price",
    "max_price"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_cols].dtypes)

temperature         float64
arrival_quantity    float64
import_quantity     float64
modal_price         float64
min_price           float64
max_price           float64
dtype: object


#### Step 15: Handle Missing Maximum Price (`max_price`)

In [33]:
# Impute max_price via grouped linear interpolation
df["max_price"] = (
    df.groupby(["market", "commodity"])["max_price"]
      .transform(lambda x: x.interpolate(method="linear"))
)

# Fallback 1: Market + commodity median
max_price_median = (
    df.groupby(["market", "commodity"])["max_price"]
      .transform("median")
)
df["max_price"] = df["max_price"].fillna(max_price_median)

# Fallback 2: Overall global median
df["max_price"] = df["max_price"].fillna(df["max_price"].median())

print(f"Remaining missing max_price: {df['max_price'].isnull().sum():,}")

Remaining missing max_price: 0


## Sub-section 1.6: Post-Imputation Quality Assurance & Intermediate Checkpoint

#### Step 16: Final Verification of Missing Values

In [34]:
# Detect missing values in each column
missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

Missing values in each column:
date                0
state               0
district            0
market              0
commodity           0
arrival_quantity    0
min_price           0
max_price           0
modal_price         0
msp                 0
import_quantity     0
export_quantity     0
import_value        0
export_value        0
fuel_cost           0
temperature         0
humidity            0
msp_applicable      0
serial_no           0
old_index           0
data_entry_user     0
dtype: int64


#### Step 17: Verify Dataset Shape & Row Count Integrity
Imputation must replace missing values without inadvertently adding or dropping any rows.

In [35]:
# Verify dimensions remain intact
print(f"Rows after missing-value cleaning    : {df.shape[0]:,}")
print(f"Columns after missing-value cleaning : {df.shape[1]}")

Rows after missing-value cleaning    : 282,504
Columns after missing-value cleaning : 21


#### Step 18: Save Intermediate Cleaned Dataset
Export the dataset to preserve our progress before moving to subsequent cleaning phases (such as duplicate handling, type casting, outlier detection, and feature engineering).

In [36]:
# Save intermediate cleaned dataset
output_file = "dataset_after_missing_value_cleaning.csv"
df.to_csv(output_file, index=False)

print(f"Missing-value cleaning successfully completed and saved to '{output_file}'!")

Missing-value cleaning successfully completed and saved to 'dataset_after_missing_value_cleaning.csv'!


---
# SECTION 2: Problem 2 – Duplicate Rows Detection, Analysis & Removal

### Strategy & Pipeline Architecture:
1. **Detection & Counting**: Identify the exact number of duplicated rows across the dataset (`df.duplicated().sum()`).
2. **Deep-Dive Inspection (`keep=False`)**: Isolate all occurrences (both original and copies) to observe the structure and frequency of duplicate records.
3. **Frequency Aggregation**: Quantify how many times complete identical records appear in the dataset.
4. **Deduplication**: Execute `drop_duplicates().reset_index(drop=True)` to eliminate redundant rows and re-index.
5. **Verification & Shape Validation**: Verify 0 remaining exact duplicates and confirm the dataset shape transition from **282,504 → 270,504 rows**.
6. **Future Business-Key Duplicate Roadmap**: Note that a secondary domain-level duplicate check (`date + market + commodity`) will be performed following text and categorical standardization.

---

## Sub-section 2.1: Duplicate Detection & Exploration

#### Step 1: Count Duplicate Rows
We begin by calculating the total count of exact duplicate records present in the dataset.

In [37]:
# ============================================================
# PROBLEM 2: DUPLICATE ROWS
# STEP 1: Detect duplicate rows
# ============================================================

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 9085


#### Step 2: Show Duplicate Records
### Why `keep=False`?
* Normally, `df.duplicated()` (with default `keep='first'`) marks only the **second and subsequent copies** as duplicates.
* By specifying `keep=False`, Pandas marks **all occurrences** (both original and duplicates) of the repeated record.
* This allows us to inspect full duplicate groups side-by-side.

In [38]:
# Show duplicate rows (all copies included)
duplicate_rows = df[df.duplicated(keep=False)]

print("Total rows involved in duplicate groups:", len(duplicate_rows))
display(duplicate_rows.head(20))

Total rows involved in duplicate groups: 18170


,date,state,district,market,commodity,arrival_quantity,min_price,max_price,modal_price,msp,...,export_quantity,import_value,export_value,fuel_cost,temperature,humidity,msp_applicable,serial_no,old_index,data_entry_user
16,2023-06-01,Maharashtra,Nagpur,BhIwApUr,Rice,915.0,5622.0,6527.0,6098.0,2040,...,4257,87.38,160.79,93.79,30.0,63.1,1,80522,80521,collector_02
17,2023-06-01,Maharashtra,Nagpur,BhIwApUr,Rice,915.0,5622.0,6527.0,6098.0,2040,...,4257,87.38,160.79,93.79,30.0,63.1,1,80522,80521,collector_02
20,2026-07-19,Maharashtra,Nagpur,BhIwApUr,Soyabean,1178.0,5994.0,6752.0,6346.0,5708,...,2497,168.31,203.68,93.43,30.0,83.1,1,259012,259011,collector_04
21,2026-07-19,Maharashtra,Nagpur,BhIwApUr,Soyabean,1178.0,5994.0,6752.0,6346.0,5708,...,2497,168.31,203.68,93.43,30.0,83.1,1,259012,259011,collector_04
49,2024-12-04,Maharashtra,Nagpur,Bhiwapur,COTTON CROP,966.0,6153.0,6923.0,6499.0,7121,...,3113,161.71,221.82,₹ 93.04,20.5,65.2,1,166621,166620,collector_01
50,2024-12-04,Maharashtra,Nagpur,Bhiwapur,COTTON CROP,966.0,6153.0,6923.0,6499.0,7121,...,3113,161.71,221.82,₹ 93.04,20.5,65.2,1,166621,166620,collector_01
55,2022-04-14,Maharashtra,Wardha,Bhiwapur,Cotton,925.0,3866.0,4528.0,4172.0,5726,...,3092,101.32,127.96,94.12,34.0,58.4,1,16081,16080,collector_01
56,2022-04-14,Maharashtra,Wardha,Bhiwapur,Cotton,925.0,3866.0,4528.0,4172.0,5726,...,3092,101.32,127.96,94.12,34.0,58.4,1,16081,16080,collector_01
92,2025-05-20,Maharashtra,Nagpur,Bhiwapur,Cotton,723.0,4222.0,5167.0,4622.0,7121,...,3159,117.49,202.92,93.71,34.7,60.6,1,192673,192672,collector_01
93,2025-05-20,Maharashtra,Nagpur,Bhiwapur,Cotton,723.0,4222.0,5167.0,4622.0,7121,...,3159,117.49,202.92,93.71,34.7,60.6,1,192673,192672,collector_01


#### Step 3: Check Duplicate Frequency by Complete Row
We aggregate identical records to evaluate the exact frequency distribution of complete duplicate rows across all columns.

In [39]:
# Find frequency of duplicate complete rows
duplicate_frequency = (
    df.groupby(list(df.columns), dropna=False)
      .size()
      .reset_index(name="count")
)

duplicate_frequency = duplicate_frequency[
    duplicate_frequency["count"] > 1
]

display(duplicate_frequency.head(10))

,date,state,district,market,commodity,arrival_quantity,min_price,max_price,modal_price,msp,...,import_value,export_value,fuel_cost,temperature,humidity,msp_applicable,serial_no,old_index,data_entry_user,count
0,2022-01-01,Maharashtra,Nagpur,Katol,watermelon,694.0,2815.0,3622.0,3145.0,0,...,42.29,29.72,93.54,20.3,58.2,0,134,133,collector_02,2
11,2022-01-01,Maharashtra,Bhandara,Kalmeshwar Mandi,Tomato,1689.0,2276.0,2697.0,2465.0,0,...,50.86,21.21,93.47,20.1,57.9,0,106,105,collector_10,2
15,2022-01-01,Maharashtra,Nagpur,Bhiwapur,Rice,1507.0,4603.0,5161.0,4875.0,1940,...,118.36,133.61,93.37,21.4,59.4,1,26,25,collector_02,2
21,2022-01-01,Maharashtra,Nagpur,Mauda,Cotton,1121.0,6278.0,7040.0,6584.0,5726,...,182.76,172.28,93.45,20.5,58.4,1,6,5,collector_06,2
41,2022-01-01,Maharashtra,Nagpur,Hingna,Mango,438.0,4443.0,4731.0,4595.0,0,...,50.55,48.65,93.49,19.9,57.7,0,144,143,collector_12,2
54,2022-01-01,Maharashtra,Nagpur,Kalmeshwar,Cotton,1078.0,4108.0,6808.0,6560.0,5726,...,168.96,223.17,93.43,20.1,57.9,1,2,1,collector_02,2
78,2022-01-01,Maharashtra,Nagpur,Mandhal,Potato,1008.0,5112.0,5439.0,5461.0,0,...,45.18,40.46,93.26,20.4,58.3,0,83,82,collector_03,2
102,2022-01-01,Maharashtra,Nagpur,Nagpur Market,Tomato,2565.0,2320.0,2645.0,2473.0,0,...,89.92,17.19,93.52,20.7,58.6,0,111,110,collector_03,2
135,2022-01-01,Maharashtra,Nagpur,Savner,Soyabean,1596.0,6605.0,7508.0,7038.0,3950,...,288.68,186.99,93.43,21.2,59.2,1,50,49,collector_02,2
156,2022-01-02,Maharashtra,Nagpur,Hingna,PeA,813.0,4048.0,4861.0,4375.0,0,...,37.91,18.89,93.73,19.1,59.4,0,209,208,collector_05,2


## Sub-section 2.2: Exact Duplicate Removal & Verification

#### Step 4: Remove Exact Duplicate Rows
We drop all exact duplicate rows, keeping only the first occurrence of each row and resetting the index to maintain a clean, continuous sequence.

In [40]:
# ============================================================
# REMOVE EXACT DUPLICATE ROWS
# ============================================================

before_rows = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after_rows = len(df)

print("Rows before removing duplicates:", before_rows)
print("Rows after removing duplicates :", after_rows)
print("Rows removed                   :", before_rows - after_rows)

Rows before removing duplicates: 282504
Rows after removing duplicates : 273419
Rows removed                   : 9085


#### Step 5: Verify That Exact Duplicates Are Gone
Ensure that the remaining duplicate count is strictly 0.

In [41]:
# ============================================================
# VERIFY DUPLICATES
# ============================================================

remaining_duplicates = df.duplicated().sum()

print("Remaining exact duplicate rows:", remaining_duplicates)

Remaining exact duplicate rows: 0


#### Step 6: Verify Dataset Size & Integrity
Verify the updated dimensions of the dataset after deduplication (expected: `270,504` rows and `21` columns).

> **Note on Column Count (21 columns):**  
> We retain all 21 columns for now; unnecessary metadata columns (`serial_no`, `old_index`, `data_entry_user`) will be dropped in their dedicated cleaning step.

> **Important Note on Business-Key Duplicates:**  
> We have eliminated **exact duplicate rows**. Later, after cleaning whitespaces, capitalization, crop names, market names, dates, and categorical values, we will perform a secondary **business-key duplicate check** on `[date, market, commodity]` to catch semantic duplicates (e.g., `"Nagpur"` vs `"NAGPUR"` vs `"Nagpur Mandi"`).

In [42]:
print("Current dataset shape:", df.shape)

Current dataset shape: (273419, 21)


---
# SECTION 3: Problem 3 – Extra Spaces Detection & Normalization

### Strategy & Pipeline Architecture:
1. **Target Identification**: Isolate all key categorical text columns (`state`, `district`, `market`, `commodity`).
2. **Leading & Trailing Space Detection**: Identify and count records where `str.strip()` differs from the raw string.
3. **Visual Inspection**: Display representative examples showing leading, trailing, and padded strings.
4. **Leading & Trailing Whitespace Stripping**: Apply `.str.strip()` across all target text columns.
5. **Internal Multi-Whitespace Collapse**: Use regex `r"\s+"` replacement to normalize internal multiple spaces (e.g., `"Green   Chilli"` $\rightarrow$ `"Green Chilli"`).
6. **Post-Cleaning Verification**: Re-evaluate space counts to confirm 0 records contain extra spaces.

> **Note on Capitalization:** Case normalization is deliberately handled in the next dedicated step (Problem 4), preserving clean separation of concerns.

---

## Sub-section 3.1: Define Affected Columns & Detect Extra Whitespaces

#### Step 1: Detect Extra Spaces
Identify text columns and calculate how many values contain leading or trailing whitespace.

In [ ]:
# ============================================================
# PROBLEM 3: EXTRA SPACES
# STEP 1: DETECT EXTRA SPACES
# ============================================================

text_columns = [
    "state",
    "district",
    "market",
    "commodity"
]

for col in text_columns:
    spaces = df[col].astype(str).str.strip() != df[col].astype(str)
    print(f"{col}: {spaces.sum()} values with leading/trailing spaces")

#### Step 2: Show Examples of Values Containing Extra Spaces
Inspect concrete examples from affected text columns to observe leading, trailing, and excessive space padding.

In [ ]:
# Show examples of values containing extra spaces

for col in text_columns:
    spaces = df[col].astype(str).str.strip() != df[col].astype(str)

    if spaces.sum() > 0:
        print(f"\nExamples from {col}:")
        display(df.loc[spaces, [col]].head(10))

## Sub-section 3.2: Whitespace Cleaning (Stripping & Regex Normalization)

#### Step 3: Remove Leading and Trailing Spaces
Strip outer whitespace from all text columns.

In [ ]:
# ============================================================
# STEP 3: REMOVE EXTRA SPACES (LEADING / TRAILING)
# ============================================================

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

#### Step 4: Remove Repeated Internal Spaces
Collapse multiple consecutive internal spaces into a single space (e.g., `"Green   Chilli"` $\rightarrow$ `"Green Chilli"`).

In [ ]:
# ============================================================
# STEP 4: COLLAPSE REPEATED INTERNAL SPACES
# ============================================================

for col in text_columns:
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)

## Sub-section 3.3: Post-Cleaning Verification

#### Step 5: Verify That Extra Spaces Are Eliminated
Ensure all text columns have 0 remaining leading/trailing whitespace inconsistencies.

In [ ]:
# ============================================================
# STEP 5: VERIFY EXTRA SPACES
# ============================================================

for col in text_columns:
    spaces = df[col].astype(str).str.strip() != df[col].astype(str)
    print(f"{col}: {spaces.sum()} values with extra spaces")

---
# SECTION 4: Problem 4 – Inconsistent Capitalization & Canonical Casing

### Strategy & Pipeline Architecture:
1. **Case Inconsistency Exploration**: Inspect unique values across categorical text columns (`state`, `district`, `market`, `commodity`) to identify irregular casing patterns (`WHEAT`, `wHeAt`, `nAgPuR`, etc.).
2. **Lowercasing Transformation**: Normalize all strings temporarily to standard lowercase (`.str.lower()`) to eliminate case discrepancies.
3. **Post-Lowercasing Inspection**: Verify that all casing variants have merged into uniform lowercase strings.
4. **Canonical Form Mapping**: Map known lowercased entities into professional Proper/Title Case standards for `commodity`, `market`, and `state` using explicit domain mapping dictionaries.
5. **District Isolation Note**: Retain `district` as-is for now, since it contains geographical inconsistencies/corrupted entries that will be resolved during the dedicated Geographical Cleaning phase.
6. **Final Verification**: Inspect sorted unique values across `commodity`, `market`, and `state` to verify clean canonical titles.

---

## Sub-section 4.1: Capitalization Inconsistency Discovery

#### Step 1: Inspect Inconsistent Unique Values Across Categorical Columns

In [ ]:
# ============================================================
# PROBLEM 4: INCONSISTENT CAPITALIZATION
# STEP 1: INSPECT INCONSISTENT CAPITALIZATION
# ============================================================

text_columns = [
    "state",
    "district",
    "market",
    "commodity"
]

for col in text_columns:
    print(f"\nUnique values in {col} before capitalization cleaning:")
    print(df[col].unique()[:30])

## Sub-section 4.2: Lowercasing Normalization & Verification

#### Step 2: Convert All Categorical Strings Temporarily to Lowercase

In [ ]:
# ============================================================
# STEP 2: CONVERT TEXT TO LOWERCASE
# ============================================================

for col in text_columns:
    df[col] = df[col].str.lower()

#### Step 3: Verify That Case Inconsistencies Are Merged Into Lowercase

In [ ]:
# Verify all values are lowercase
for col in text_columns:
    print(f"\n{col}:")
    print(df[col].unique()[:30])

## Sub-section 4.3: Canonical Standard Name Mapping & Verification

#### Step 4: Map Lowercase Values to Project Canonical Standard Names

> **Important Note on `district`:**  
> We deliberately defer `district` casing/mapping to the upcoming **Geographical Information Cleaning** phase, because the dataset contains corrupted/incorrect district values that require validation rather than naive capitalization.

In [ ]:
# ============================================================
# STEP 4: CANONICAL NAMES MAPPING
# ============================================================

crop_mapping = {
    "cotton": "Cotton",
    "rice": "Rice",
    "wheat": "Wheat",
    "soyabean": "Soyabean",
    "soybean": "Soyabean",
    "pea": "Pea",
    "orange": "Orange",
    "potato": "Potato",
    "onion": "Onion",
    "tomato": "Tomato",
    "green chilli": "Green Chilli",
    "watermelon": "Watermelon",
    "mango": "Mango"
}

market_mapping = {
    "hingna": "Hingna",
    "kalmeshwar": "Kalmeshwar",
    "kamthi": "Kamthi",
    "katol": "Katol",
    "mandhal": "Mandhal",
    "mauda": "Mauda",
    "nagpur": "Nagpur",
    "narkhed": "Narkhed",
    "parshivni": "Parshivni",
    "ramtek": "Ramtek",
    "savner": "Savner",
    "umred": "Umred",
    "bhiwapur": "Bhiwapur"
}

state_mapping = {
    "maharashtra": "Maharashtra"
}

# Apply canonical mappings
df["commodity"] = df["commodity"].replace(crop_mapping)
df["market"] = df["market"].replace(market_mapping)
df["state"] = df["state"].replace(state_mapping)

#### Step 5: Verify Final Canonical Names

> **Note on Spelling Variants:**  
> While capitalization is now standardized, the upcoming **Problem 5** step will handle spelling variations and aliases (such as `soy bean`, `green chili`, etc.).

In [ ]:
# ============================================================
# STEP 5: VERIFY CANONICAL NAMES
# ============================================================

print("Crops:")
print(sorted(df["commodity"].dropna().unique()))

print("\nMarkets:")
print(sorted(df["market"].dropna().unique()))

print("\nStates:")
print(df["state"].dropna().unique())

---
# SECTION 5: Problem 5 – Inconsistent Crop Names (`commodity`) Standardization

### Strategy & Pipeline Architecture:
1. **Discovery & Count**: Count initial unique crop names and display the raw list of variants.
2. **Text Normalization**: Create a temporary normalized column `commodity_clean` by converting to lowercase, stripping whitespace, and removing punctuation/separators (`[\s_-]+`).
3. **Canonical Variant Dictionary**: Define an explicit domain dictionary mapping spelling variants (`soybean`, `greenchili`, `greenchilly`, etc.) to standard canonical names (`Soyabean`, `Green Chilli`, etc.).
4. **Safe Mapping & Unmapped Audit**: Apply `.map(crop_mapping)` and strictly inspect any unmapped (`NaN`) values to prevent silent data corruption or hallucinated mappings.
5. **Cleanup Temporary Artifacts**: Drop the helper column `commodity_clean`.
6. **Final Verification**: Confirm that exactly 12 standard crops exist with 0 missing/unmapped entries.

---

## Sub-section 5.1: Crop Name Inconsistency Discovery

#### Step 1: Check Current Crop Names and Unique Count

In [ ]:
# ============================================================
# PROBLEM 5: INCONSISTENT CROP NAMES
# STEP 1: CHECK UNIQUE VALUES
# ============================================================

print("Number of unique crop names before cleaning:",
      df["commodity"].nunique())

print("\nCrop names:")
print(sorted(df["commodity"].dropna().unique()))

## Sub-section 5.2: String Normalization for Variant Matching

#### Step 2: Create a Normalized Helper Column
Strip spaces, hyphens, and underscores to simplify variant identification (e.g., `"Soy Bean"` $\rightarrow$ `"soybean"`, `"SOY-BEAN"` $\rightarrow$ `"soybean"`).

In [ ]:
# ============================================================
# STEP 2: NORMALIZE CROP TEXT
# ============================================================

df["commodity_clean"] = (
    df["commodity"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"[\s_-]+", "", regex=True)
)

## Sub-section 5.3: Canonical Crop Mapping & Unmapped Audit

#### Step 3: Define Canonical Crop Mapping & Apply to Dataset

In [ ]:
# ============================================================
# STEP 3: MAP ALL CROP VARIANTS
# ============================================================

crop_mapping = {
    "cotton": "Cotton",
    "rice": "Rice",
    "wheat": "Wheat",

    "soyabean": "Soyabean",
    "soybean": "Soyabean",

    "pea": "Pea",

    "orange": "Orange",

    "potato": "Potato",

    "onion": "Onion",

    "tomato": "Tomato",

    "greenchilli": "Green Chilli",
    "greenchili": "Green Chilli",
    "greenchilly": "Green Chilli",

    "watermelon": "Watermelon",
    "mango": "Mango"
}

df["commodity"] = df["commodity_clean"].map(crop_mapping)

#### Step 4: Audit and Check for Unmapped Crop Values

> **Important:** Never use blind fuzzy matching or drop unmapped values silently. Any unmapped values must be inspected and explicitly appended to `crop_mapping`.

In [ ]:
# ============================================================
# STEP 4: CHECK UNMAPPED CROP VALUES
# ============================================================

unmapped = df[df["commodity"].isna()]["commodity_clean"].unique()

print("Number of unmapped crop values:", len(unmapped))

print("\nUnmapped values:")
print(unmapped)

# If unmapped values exist, update crop_mapping and re-apply
if len(unmapped) > 0:
    crop_mapping.update({
        "greenchillies": "Green Chilli",
        "soybeans": "Soyabean"
    })
    df["commodity"] = df["commodity_clean"].map(crop_mapping)

## Sub-section 5.4: Cleanup & Final Verification

#### Step 5: Remove Temporary Helper Column

In [ ]:
# ============================================================
# STEP 5: REMOVE TEMPORARY COLUMN
# ============================================================

df.drop(columns=["commodity_clean"], inplace=True)

#### Step 6: Final Verification of Standard Crop Names
Confirm that exactly 12 standard crops remain.

In [ ]:
# ============================================================
# STEP 6: VERIFY CROP NAMES
# ============================================================

print("Number of unique crop names after cleaning:",
      df["commodity"].nunique())

print("\nFinal crop names:")
print(sorted(df["commodity"].unique()))

---
# SECTION 6: Problem 6 – Inconsistent Mandi / Market Names (`market`) Standardization

### Strategy & Pipeline Architecture:
1. **Discovery & Inspection**: Profile the count and unique values of market entries before cleaning.
2. **Text Normalization & Suffix Stripping**: Create a temporary normalized helper column `market_clean` by lowercasing, trimming whitespace, removing common trading entity suffixes (`mandi`, `apmc`, `market`), and removing separators (`[\s_-]+`).
3. **Canonical Market Mapping**: Define an explicit dictionary mapping normalized tokens to the 13 canonical APMC market centers in Nagpur district.
4. **Unmapped Values Audit**: Audit unmapped (`NaN`) values to prevent unhandled abbreviations or corrupted names.
5. **Cleanup Helper Columns**: Drop `market_clean`.
6. **Final Verification**: Confirm exactly 13 canonical markets remain with 0 missing entries.

---

## Sub-section 6.1: Market Name Discovery & Profiling

#### Step 1: Check Current Market Names and Unique Count

In [ ]:
# ============================================================
# PROBLEM 6: INCONSISTENT MARKET NAMES
# STEP 1: CHECK UNIQUE VALUES
# ============================================================

print("Number of unique market names before cleaning:",
      df["market"].nunique())

print("\nMarket names:")
print(sorted(df["market"].dropna().unique()))

## Sub-section 6.2: Market Text Normalization & Suffix Removal

#### Step 2: Normalize Strings & Strip Suffixes (`mandi`, `apmc`, `market`)

In [ ]:
# ============================================================
# STEP 2: NORMALIZE MARKET NAMES
# ============================================================

df["market_clean"] = (
    df["market"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Remove common suffixes and whitespace/hyphen/underscore separators
df["market_clean"] = (
    df["market_clean"]
    .str.replace(r"\b(mandi|apmc|market)\b", "", regex=True)
    .str.replace(r"[\s_-]+", "", regex=True)
)

## Sub-section 6.3: Canonical Market Mapping & Unmapped Audit

#### Step 3: Define Canonical Market Mapping Dictionary & Apply

In [ ]:
# ============================================================
# STEP 3: CANONICAL MARKET MAPPING
# ============================================================

market_mapping = {
    "hingna": "Hingna",
    "kalmeshwar": "Kalmeshwar",
    "kamthi": "Kamthi",
    "katol": "Katol",
    "mandhal": "Mandhal",
    "mauda": "Mauda",
    "nagpur": "Nagpur",
    "narkhed": "Narkhed",
    "parshivni": "Parshivni",
    "ramtek": "Ramtek",
    "savner": "Savner",
    "umred": "Umred",
    "bhiwapur": "Bhiwapur"
}

df["market"] = df["market_clean"].map(market_mapping)

#### Step 4: Audit Unmapped Market Values

> **Important:** Stop and inspect any unmapped entries to ensure all spelling/abbreviation variants are accounted for before proceeding.

In [ ]:
# ============================================================
# STEP 4: CHECK UNMAPPED VALUES
# ============================================================

unmapped_markets = (
    df[df["market"].isna()]["market_clean"]
    .unique()
)

print("Number of unmapped market values:",
      len(unmapped_markets))

print("\nUnmapped market values:")
print(unmapped_markets)

## Sub-section 6.4: Cleanup & Final Verification

#### Step 5: Remove Temporary Helper Column

In [ ]:
# ============================================================
# STEP 5: REMOVE TEMPORARY COLUMN
# ============================================================

df.drop(columns=["market_clean"], inplace=True)

#### Step 6: Final Verification of Standard Market Names
Confirm that exactly 13 standard markets remain.

In [ ]:
# ============================================================
# STEP 6: VERIFY MARKET NAMES
# ============================================================

print("Number of unique market names after cleaning:",
      df["market"].nunique())

print("\nFinal market names:")
print(sorted(df["market"].unique()))

---
# SECTION 7: Problem 7 – Geographical Information (`state` & `district`) Standardization

### Strategy & Pipeline Architecture:
1. **Discovery & Exploration**: Profile initial unique state and district values to identify noise, spelling variations, and out-of-scope regional labels.
2. **State Normalization & Canonical Mapping**: Normalize `state` via string cleaning (`state_clean`) and map all spelling/alias variations to `"Maharashtra"`.
3. **State Unmapped Audit**: Check for any unmapped state values.
4. **District Normalization & Project-Scope Alignment**: Normalize `district` (`district_clean`) and align all district variants and synthetic boundary deviations to the project-defined canonical `"Nagpur"`.
5. **District Unmapped Audit**: Verify 0 unmapped district entries.
6. **Cleanup & Final Verification**: Drop helper columns and confirm exactly 1 state (`'Maharashtra'`) and 1 district (`'Nagpur'`).

> **Domain Scope Note:** Under the defined scope of this project, all 13 agricultural markets belong to the Nagpur agricultural market district dataset.

---

## Sub-section 7.1: Geographical Discovery & Profiling

#### Step 1: Check Current State and District Values

In [ ]:
# ============================================================
# PROBLEM 7: GEOGRAPHICAL INFORMATION
# STEP 1: CHECK CURRENT VALUES
# ============================================================

print("Unique state values:")
print(sorted(df["state"].dropna().unique()))

print("\nNumber of unique state values:",
      df["state"].nunique())

print("\nUnique district values:")
print(sorted(df["district"].dropna().unique()))

print("\nNumber of unique district values:",
      df["district"].nunique())

## Sub-section 7.2: State Normalization & Mapping

#### Step 2: Normalize and Map State to Canonical Form

In [ ]:
# ============================================================
# STEP 2: NORMALIZE STATE
# ============================================================

df["state_clean"] = (
    df["state"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"[\s_-]+", " ", regex=True)
)

state_mapping = {
    "maharashtra": "Maharashtra",
    "maharastra": "Maharashtra",
    "maharahstra": "Maharashtra",
    "maharashtra state": "Maharashtra",
    "mh state": "Maharashtra"
}

df["state"] = df["state_clean"].map(state_mapping)

#### Step 3: Audit Unmapped State Values

In [ ]:
unmapped_states = df[df["state"].isna()]["state_clean"].unique()

print("Unmapped state values:")
print(unmapped_states)

## Sub-section 7.3: District Normalization & Project-Scope Mapping

#### Step 4: Normalize District Text

In [ ]:
# ============================================================
# STEP 4: NORMALIZE DISTRICT
# ============================================================

df["district_clean"] = (
    df["district"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"[\s_-]+", " ", regex=True)
)

#### Step 5: Map District Values to Project Canonical Geography (`Nagpur`)

In [ ]:
district_mapping = {
    "nagpur": "Nagpur",
    "nagpur district": "Nagpur",
    "nagpur rural": "Nagpur",
    "wardha": "Nagpur",
    "wardha district": "Nagpur",
    "bhandara": "Nagpur",
    "amravati": "Nagpur",
    "chandrapur": "Nagpur"
}

df["district"] = df["district_clean"].map(district_mapping)

#### Step 6: Audit Unmapped District Values

> **Important:** Audit unmapped district entries to verify that all observed variations map correctly to the defined canonical district.

In [ ]:
unmapped_districts = (
    df[df["district"].isna()]["district_clean"]
    .unique()
)

print("Number of unmapped district values:",
      len(unmapped_districts))

print("\nUnmapped district values:")
print(unmapped_districts)

## Sub-section 7.4: Cleanup & Final Verification

#### Step 7: Remove Temporary Helper Columns

In [ ]:
df.drop(
    columns=["state_clean", "district_clean"],
    inplace=True
)

#### Step 8: Final Verification of Geographical Consistency
Confirm dataset contains strictly 1 state and 1 district.

In [ ]:
# ============================================================
# FINAL GEOGRAPHICAL VALIDATION
# ============================================================

print("States:")
print(df["state"].unique())

print("\nDistricts:")
print(df["district"].unique())

print("\nNumber of states:", df["state"].nunique())
print("Number of districts:", df["district"].nunique())

---
# SECTION 8: Problem 8 – Special Characters in Numeric Price Columns & Type Casting

### Strategy & Pipeline Architecture:
1. **Discovery & Inspection**: Inspect string formatting in price columns (`min_price`, `modal_price`, `max_price`) to identify currency signs (`₹`, `INR`), thousands separators (`,`), and suffix artifacts (`/-`).
2. **Special Character Stripping**: Clean all formatting artifacts using chained string replacements.
3. **Non-Numeric Value Audit**: Audit any remaining non-numeric entries using `pd.to_numeric(..., errors='coerce')` to ensure no corrupted strings are silently erased.
4. **Type Conversion to Numeric**: Cast all cleaned price columns explicitly to `float64`.
5. **Post-Conversion Validation**: Verify column `dtypes` and print sample numeric values.

> **Roadmap Note:** This step strictly handles syntax/formatting cleanup and type casting. Domain-level unit standardizations (e.g., `₹/kg` vs `₹/quintal`), logical price hierarchy checks (`min_price <= modal_price <= max_price`), and statistical outlier handling will occur in subsequent dedicated sections.

---

## Sub-section 8.1: Price Formatting Discovery

#### Step 1: Inspect Raw String Values in Price Columns

In [ ]:
# ============================================================
# PROBLEM 8: SPECIAL CHARACTERS IN NUMERIC VALUES
# STEP 1: SHOW CURRENT VALUES
# ============================================================

price_columns = [
    "min_price",
    "modal_price",
    "max_price"
]

for col in price_columns:
    print(f"\nExamples from {col}:")
    print(df[col].dropna().astype(str).head(20).tolist())

## Sub-section 8.2: Formatting & Special Character Stripping

#### Step 2: Remove Currency Symbols, Commas, and Suffixes

In [ ]:
# ============================================================
# STEP 2: REMOVE SPECIAL CHARACTERS
# ============================================================

for col in price_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace("₹", "", regex=False)
        .str.replace("INR", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("/-", "", regex=False)
        .str.strip()
    )

## Sub-section 8.3: Non-Numeric Audit & Data Type Casting

#### Step 3: Audit Any Remaining Non-Numeric Values

In [ ]:
# ============================================================
# STEP 3: CHECK NON-NUMERIC VALUES
# ============================================================

for col in price_columns:
    numeric_check = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    invalid = df.loc[
        numeric_check.isna() & df[col].notna(),
        col
    ]

    print(f"\n{col} - remaining non-numeric values:")
    print(invalid.unique()[:20])

#### Step 4: Convert Price Columns to Numeric (`float64`)

In [ ]:
# ============================================================
# STEP 4: CONVERT PRICE COLUMNS TO NUMERIC
# ============================================================

for col in price_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

## Sub-section 8.4: Validation & Quality Check

#### Step 5: Verify Data Types Across Price Columns

In [ ]:
# ============================================================
# STEP 5: VERIFY DATA TYPES
# ============================================================

print(df[price_columns].dtypes)

#### Step 6: Verify Sample Values After Conversion

In [ ]:
for col in price_columns:
    print(f"\n{col} after cleaning:")
    print(df[col].head(10).tolist())

---
# SECTION 9: Problem 9 – Inconsistent Units Detection & Normalization (₹/kg $\rightarrow$ ₹/quintal)

### Strategy & Pipeline Architecture:
1. **Standard Unit Definition**: Agricultural commodity mandis operate in **₹ per quintal** ($1\text{ quintal} = 100\text{ kg}$). Records entered in ₹/kg must be converted by a scaling factor of 100.
2. **Identification & Discovery**: Scan string representations across `min_price`, `modal_price`, and `max_price` using regex `r"/kg|per kg|kg"` to count records with explicit per-kg units.
3. **Visual Inspection**: Display sample rows showing unnormalized per-kg pricing.
4. **Conversion & Scaling**: Isolate identified per-kg records, extract the numerical rate using regex `r"(\d+(?:\.\d+)?)"`, cast to float, and scale by $\times 100$.
5. **Post-Normalization Verification**: Ensure 0 remaining records contain kg-unit indicators and verify price consistency.

> **Domain Rule Note:** Scaling by 100 is applied strictly to records explicitly tagged with per-kg units, avoiding unintended dataset-wide inflation.

---

## Sub-section 9.1: Unit Discrepancy Discovery

#### Step 1: Check for Unit-Related Entries Across Price Columns

In [ ]:
# ============================================================
# PROBLEM 9: INCONSISTENT UNITS
# STEP 1: CHECK FOR UNIT INFORMATION
# ============================================================

price_columns = [
    "min_price",
    "modal_price",
    "max_price"
]

for col in price_columns:
    unit_values = df[col].astype(str).str.contains(
        r"/kg|per kg|kg",
        case=False,
        regex=True,
        na=False
    )

    print(f"{col}: {unit_values.sum()} values containing kg")

#### Step 2: Display Examples of Records Containing Per-Kg Pricing

In [ ]:
for col in price_columns:

    unit_values = df[col].astype(str).str.contains(
        r"/kg|per kg|kg",
        case=False,
        regex=True,
        na=False
    )

    if unit_values.sum() > 0:
        print(f"\nExamples from {col}:")
        display(df.loc[unit_values, [col]].head(20))

## Sub-section 9.2: Unit Conversion & Scaling (₹/kg $\rightarrow$ ₹/quintal)

#### Step 3: Convert ₹/kg Values to ₹/quintal ($\times 100$)

In [ ]:
# ============================================================
# STEP 3: CONVERT KG VALUES TO QUINTAL
# ============================================================

for col in price_columns:

    kg_mask = df[col].astype(str).str.contains(
        r"/kg|per kg|kg",
        case=False,
        regex=True,
        na=False
    )

    # Extract numeric value
    kg_values = (
        df.loc[kg_mask, col]
        .astype(str)
        .str.extract(r"(\d+(?:\.\d+)?)")[0]
    )

    kg_values = pd.to_numeric(
        kg_values,
        errors="coerce"
    )

    # 1 quintal = 100 kg
    df.loc[kg_mask, col] = kg_values * 100

## Sub-section 9.3: Verification & Unit Consistency Audit

#### Step 4: Verify Price Columns After Unit Normalization

In [ ]:
print("Price columns after unit normalization:")
print(df[price_columns].head(20))

print("\nVerify remaining kg values:")
for col in price_columns:

    remaining = df[col].astype(str).str.contains(
        r"/kg|per kg|kg",
        case=False,
        regex=True,
        na=False
    ).sum()

    print(f"{col}: remaining kg values = {remaining}")

---
# SECTION 10: Problem 10 – Column Data Types Conversion & Schema Harmonization

### Expected Data Schema:
| Column | Target Data Type | Semantic Category |
| :--- | :--- | :--- |
| `date` | `datetime64[ns]` | Temporal Index |
| `state`, `district`, `market`, `commodity` | `string` | Categorical Text |
| `arrival_quantity`, `import_quantity`, `export_quantity` | `float64` | Volume / Supply Metrics |
| `min_price`, `modal_price`, `max_price`, `msp` | `float64` | Price Metrics (₹/quintal) |
| `import_value`, `export_value`, `fuel_cost` | `float64` | Trade & Transport Economic Metrics |
| `temperature`, `humidity` | `float64` | Weather Indicators |
| `msp_applicable` | `Int64` | Binary Policy Flag (0/1) |

---

## Sub-section 10.1: Initial Schema Audit

#### Step 1: Check Current Data Types Across All Columns

In [ ]:
# ============================================================
# PROBLEM 10: WRONG DATA TYPES
# STEP 1: CHECK CURRENT DATA TYPES
# ============================================================

print("Current data types:")
print(df.dtypes)

## Sub-section 10.2: Quantitative Features Conversion

#### Step 2 & 3: Define & Convert Numeric Columns to Numeric (`float64`)
Use `pd.to_numeric(..., errors='coerce')` so that any residual malformed strings safely coerce to `NaN` rather than triggering execution errors.

In [ ]:
# ============================================================
# STEP 2 & 3: CONVERT NUMERIC COLUMNS
# ============================================================

numeric_columns = [
    "arrival_quantity",
    "min_price",
    "max_price",
    "modal_price",
    "msp",
    "import_quantity",
    "export_quantity",
    "import_value",
    "export_value",
    "fuel_cost",
    "temperature",
    "humidity"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

## Sub-section 10.3: Categorical & Indicator Features Conversion

#### Step 4: Convert Categorical Text Columns to Pandas `string` Type

In [ ]:
# ============================================================
# STEP 4: CONVERT TEXT COLUMNS
# ============================================================

text_columns = [
    "state",
    "district",
    "market",
    "commodity"
]

for col in text_columns:
    df[col] = df[col].astype("string")

#### Step 5: Convert Indicator Flag `msp_applicable` to Nullable Integer (`Int64`)
Using nullable `Int64` accommodates missing or corrupted entries during intermediate stages.

In [ ]:
# ============================================================
# STEP 5: CONVERT MSP APPLICABLE
# ============================================================

df["msp_applicable"] = pd.to_numeric(
    df["msp_applicable"],
    errors="coerce"
).astype("Int64")

## Sub-section 10.4: Final Schema Verification

#### Step 6: Verify Data Types Across All Subsets

In [ ]:
# ============================================================
# STEP 6: FINAL DATA TYPE CHECK
# ============================================================

print("Data types after conversion:")
print(df.dtypes)

print("\nNumeric columns:")
print(df[numeric_columns].dtypes)

print("\nText columns:")
print(df[text_columns].dtypes)

print("\nMSP applicable:")
print(df["msp_applicable"].dtype)

---
# SECTION 11: Problem 11 – Domain-Rule Constraint Validation & Invalid Values Sanitization

### Business Domain Constraints:
| Metric Category | Target Column(s) | Valid Domain Condition | Invalid Triggers |
| :--- | :--- | :--- | :--- |
| **Weather** | `temperature` | $-10^\circ\text{C} \le \text{temp} \le 50^\circ\text{C}$ | $> 50^\circ\text{C}$ or $< -10^\circ\text{C}$ |
| **Weather** | `humidity` | $0\% \le \text{humidity} \le 100\%$ | $< 0\%$ or $> 100\%$ |
| **Policy Indicator** | `msp_applicable` | $\in \{0, 1\}$ | $\notin \{0, 1\}$ (e.g. 2, -1) |
| **Supply & Trade Volumes** | `arrival_quantity`, `import_quantity`, `export_quantity` | $\ge 0$ | $< 0$ |
| **Economic Rates** | `fuel_cost` | $\ge 0$ | $< 0$ |
| **Prices** | `min_price`, `modal_price`, `max_price` | $> 0$ | $\le 0$ |

### Strategy:
* **Do NOT drop entire rows**: Dropping complete rows causes data loss on valid co-occurring features.
* **Convert Violations to `np.nan`**: Convert domain violations to missing values (`np.nan`) so they can be accurately re-estimated using domain-specific imputation models.

---

## Sub-section 11.1: Detection & Profiling of Out-of-Bound Values

#### Step 1: Detect Invalid Entries Across Weather, Policy, Supply & Transport Columns

In [ ]:
# ============================================================
# PROBLEM 11: INVALID VALUES
# STEP 1: DETECT INVALID VALUES
# ============================================================

print("Invalid temperature values:")
print(df[(df["temperature"] > 50) | (df["temperature"] < -10)]
      [["date", "market", "commodity", "temperature"]].head(20))

print("\nInvalid humidity values:")
print(df[(df["humidity"] < 0) | (df["humidity"] > 100)]
      [["date", "market", "commodity", "humidity"]].head(20))

print("\nInvalid MSP applicable values:")
print(df[~df["msp_applicable"].isin([0, 1])]
      [["date", "market", "commodity", "msp_applicable"]].head(20))

print("\nInvalid arrival quantities:")
print(df[df["arrival_quantity"] < 0]
      [["date", "market", "commodity", "arrival_quantity"]].head(20))

print("\nInvalid import quantities:")
print(df[df["import_quantity"] < 0]
      [["date", "market", "commodity", "import_quantity"]].head(20))

print("\nInvalid export quantities:")
print(df[df["export_quantity"] < 0]
      [["date", "market", "commodity", "export_quantity"]].head(20))

print("\nInvalid fuel costs:")
print(df[df["fuel_cost"] < 0]
      [["date", "market", "commodity", "fuel_cost"]].head(20))

#### Step 2: Detect Non-Positive ($\le 0$) Price Violations

In [ ]:
price_columns = [
    "min_price",
    "modal_price",
    "max_price"
]

for col in price_columns:
    invalid = df[df[col] <= 0]
    print(f"\nInvalid values in {col}: {len(invalid)}")

    if len(invalid) > 0:
        display(
            invalid[
                ["date", "market", "commodity", col]
            ].head(20)
        )

## Sub-section 11.2: Quantitative Audit & Comprehensive Violation Metrics

#### Step 3: Count and Report All Domain Rule Violations

In [ ]:
# ============================================================
# STEP 3: COUNT INVALID VALUES
# ============================================================

invalid_counts = {
    "temperature": ((df["temperature"] > 50) |
                    (df["temperature"] < -10)).sum(),

    "humidity": ((df["humidity"] < 0) |
                 (df["humidity"] > 100)).sum(),

    "msp_applicable": (~df["msp_applicable"].isin([0, 1])).sum(),

    "arrival_quantity": (df["arrival_quantity"] < 0).sum(),

    "import_quantity": (df["import_quantity"] < 0).sum(),

    "export_quantity": (df["export_quantity"] < 0).sum(),

    "fuel_cost": (df["fuel_cost"] < 0).sum(),

    "min_price": (df["min_price"] <= 0).sum(),

    "modal_price": (df["modal_price"] <= 0).sum(),

    "max_price": (df["max_price"] <= 0).sum()
}

print("Invalid value counts:")
for column, count in invalid_counts.items():
    print(f"{column}: {count}")

## Sub-section 11.3: Sanitization & NaN Replacement

#### Step 4: Replace Invalid Values With NaN for Downstream Imputation

> **Rationale:** Marking corrupted observations as `np.nan` protects dataset sample size while enabling structured imputation.

In [ ]:
# ============================================================
# STEP 4: REPLACE INVALID VALUES WITH NaN
# ============================================================

import numpy as np

df.loc[
    (df["temperature"] > 50) | (df["temperature"] < -10),
    "temperature"
] = np.nan

df.loc[
    (df["humidity"] < 0) | (df["humidity"] > 100),
    "humidity"
] = np.nan

df.loc[
    ~df["msp_applicable"].isin([0, 1]),
    "msp_applicable"
] = np.nan

df.loc[
    df["arrival_quantity"] < 0,
    "arrival_quantity"
] = np.nan

df.loc[
    df["import_quantity"] < 0,
    "import_quantity"
] = np.nan

df.loc[
    df["export_quantity"] < 0,
    "export_quantity"
] = np.nan

df.loc[
    df["fuel_cost"] < 0,
    "fuel_cost"
] = np.nan

for col in price_columns:
    df.loc[df[col] <= 0, col] = np.nan

## Sub-section 11.4: Post-Sanitization Verification

#### Step 5: Verify That 0 Invalid Values Remain Across All Fields

In [ ]:
# ============================================================
# STEP 5: VERIFY INVALID VALUES SANITIZED
# ============================================================

print("Invalid temperature values:",
      ((df["temperature"] > 50) |
       (df["temperature"] < -10)).sum())

print("Invalid humidity values:",
      ((df["humidity"] < 0) |
       (df["humidity"] > 100)).sum())

print("Invalid MSP applicable values:",
      (~df["msp_applicable"].isin([0, 1])).sum())

print("Invalid arrival quantities:",
      (df["arrival_quantity"] < 0).sum())

print("Invalid import quantities:",
      (df["import_quantity"] < 0).sum())

print("Invalid export quantities:",
      (df["export_quantity"] < 0).sum())

print("Invalid fuel costs:",
      (df["fuel_cost"] < 0).sum())

for col in price_columns:
    print(f"Invalid {col} values:", (df[col] <= 0).sum())

---
# SECTION 12: Problem 12 – Physically & Domain Impossible Values Detection & Sanitization

### Theoretical Distinction:
* **Invalid Values (Problem 11)**: Values violating specific dataset parameter ranges or schema specifications.
* **Impossible Values (Problem 12)**: Extreme anomalies that violate fundamental physical laws or real-world physical constraints (e.g., atmospheric temperatures $<-50^\circ\text{C}$ or $>60^\circ\text{C}$, negative relative humidity, negative crop arrival volumes, negative transport fuel rates).

### Strategy:
* Document and isolate extreme physical anomalies.
* Convert identified physical anomalies to `np.nan` so that complete rows are preserved for intelligent domain imputation.

---

## Sub-section 12.1: Physical Anomaly Discovery & Inspection

#### Step 1: Detect and Display Physically Impossible Records

In [ ]:
# ============================================================
# PROBLEM 12: IMPOSSIBLE VALUES
# STEP 1: DETECT IMPOSSIBLE VALUES
# ============================================================

print("Impossible temperature values:")
display(
    df[
        (df["temperature"] < -50) |
        (df["temperature"] > 60)
    ][
        ["date", "market", "commodity", "temperature"]
    ].head(20)
)

print("\nImpossible humidity values:")
display(
    df[
        (df["humidity"] < 0) |
        (df["humidity"] > 100)
    ][
        ["date", "market", "commodity", "humidity"]
    ].head(20)
)

print("\nImpossible arrival quantities:")
display(
    df[
        df["arrival_quantity"] < 0
    ][
        ["date", "market", "commodity", "arrival_quantity"]
    ].head(20)
)

print("\nImpossible fuel costs:")
display(
    df[
        df["fuel_cost"] < 0
    ][
        ["date", "market", "commodity", "fuel_cost"]
    ].head(20)
)

## Sub-section 12.2: Quantitative Profiling

#### Step 2: Count Impossible Values

In [ ]:
# ============================================================
# STEP 2: COUNT IMPOSSIBLE VALUES
# ============================================================

print(
    "Impossible temperature values:",
    ((df["temperature"] < -50) |
     (df["temperature"] > 60)).sum()
)

print(
    "Impossible humidity values:",
    ((df["humidity"] < 0) |
     (df["humidity"] > 100)).sum()
)

print(
    "Impossible arrival quantities:",
    (df["arrival_quantity"] < 0).sum()
)

print(
    "Impossible fuel costs:",
    (df["fuel_cost"] < 0).sum()
)

## Sub-section 12.3: Safe Nullification & Verification

#### Step 3: Replace Impossible Values With NaN

In [ ]:
# ============================================================
# STEP 3: REPLACE IMPOSSIBLE VALUES WITH NaN
# ============================================================

df.loc[
    (df["temperature"] < -50) |
    (df["temperature"] > 60),
    "temperature"
] = np.nan

df.loc[
    (df["humidity"] < 0) |
    (df["humidity"] > 100),
    "humidity"
] = np.nan

df.loc[
    df["arrival_quantity"] < 0,
    "arrival_quantity"
] = np.nan

df.loc[
    df["fuel_cost"] < 0,
    "fuel_cost"
] = np.nan

#### Step 4: Final Verification of Physical Validity

In [ ]:
# ============================================================
# STEP 4: VERIFY IMPOSSIBLE VALUES REMOVED
# ============================================================

print(
    "Remaining impossible temperature values:",
    ((df["temperature"] < -50) |
     (df["temperature"] > 60)).sum()
)

print(
    "Remaining impossible humidity values:",
    ((df["humidity"] < 0) |
     (df["humidity"] > 100)).sum()
)

print(
    "Remaining impossible arrival quantities:",
    (df["arrival_quantity"] < 0).sum()
)

print(
    "Remaining impossible fuel costs:",
    (df["fuel_cost"] < 0).sum()
)

---
# SECTION 13: Problem 13 – Grouped Interquartile Range (IQR) Outlier Detection

### Statistical & Domain Architecture:
1. **Agricultural Price Heterogeneity**: Different crops (e.g., high-value Cotton vs perishable Tomato vs staple Wheat) exhibit vastly distinct price and volume distributions. Furthermore, agricultural prices fluctuate seasonally across months.
2. **Grouped IQR Formulation**: Rather than computing a naive global IQR, we compute stratified quantiles grouped by **`(commodity, month)`**:
   $$\text{IQR} = Q_3(\text{commodity, month}) - Q_1(\text{commodity, month})$$
   $$\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}, \quad \text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$$
3. **Flagging vs Deletion Policy**: We **do NOT** naively drop statistical outliers. In agricultural markets, extreme values may represent genuine supply shocks or festival demand peaks. Outliers are flagged for domain inspection and cross-feature validation before corrective decisions are made in subsequent stages.

---

## Sub-section 13.1: Target Features & Temporal Stratification

#### Step 1: Define Numerical Features for Outlier Profiling

In [ ]:
# ============================================================
# PROBLEM 13: OUTLIER DETECTION
# STEP 1: DEFINE COLUMNS
# ============================================================

outlier_columns = [
    "arrival_quantity",
    "min_price",
    "modal_price",
    "max_price"
]

#### Step 2: Extract Seasonal Month Feature for Grouped Analysis

In [ ]:
# ============================================================
# STEP 2: CREATE MONTH COLUMN
# ============================================================

df["month"] = df["date"].dt.month

print(df[["date", "month"]].head())

## Sub-section 13.2: Grouped IQR Computation & Outlier Flagging

#### Step 3: Compute Stratified IQR Bounds & Flag Outliers per Feature

In [ ]:
# ============================================================
# STEP 3: IQR OUTLIER DETECTION
# ============================================================

for col in outlier_columns:

    Q1 = df.groupby(
        ["commodity", "month"]
    )[col].transform("quantile", 0.25)

    Q3 = df.groupby(
        ["commodity", "month"]
    )[col].transform("quantile", 0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[f"{col}_outlier"] = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    )

    print(
        f"{col}:",
        df[f"{col}_outlier"].sum(),
        "potential outliers"
    )

#### Step 4: Display Representative Outlier Examples Across Target Columns

In [ ]:
# ============================================================
# STEP 4: DISPLAY OUTLIER EXAMPLES
# ============================================================

for col in outlier_columns:

    outliers = df[
        df[f"{col}_outlier"] == True
    ]

    print(f"\nExamples of {col} outliers:")

    display(
        outliers[
            [
                "date",
                "market",
                "commodity",
                col
            ]
        ].head(10)
    )

## Sub-section 13.3: Aggregate Outlier Metrics & Summary

#### Step 5: Aggregate Rows Flagged With Any Potential Outlier

In [ ]:
# ============================================================
# STEP 5: ANY OUTLIER
# ============================================================

outlier_flags = [
    f"{col}_outlier"
    for col in outlier_columns
]

df["any_outlier"] = df[outlier_flags].any(axis=1)

print(
    "Total rows containing at least one potential outlier:",
    df["any_outlier"].sum()
)

#### Step 6: Comprehensive Outlier Summary by Column

> **Note:** Retain the helper indicator flags (`month`, `*_outlier`, `any_outlier`) for the subsequent **Incorrect Data Entry** inspection phase.

In [ ]:
print("\nOutlier summary:")

for col in outlier_columns:
    print(
        f"{col}:",
        df[f"{col}_outlier"].sum()
    )

---
# SECTION 14: Problem 14 – Incorrect Data Entry & Typographical Scale Anomaly Detection

### Theoretical & Domain Framework:
1. **Data Entry Errors vs Outliers**: Unlike legitimate market volatility, data entry errors often manifest as extra zeros (order-of-magnitude decimal shifts such as $2500 \rightarrow 25000$), transcription typos, or swapped digits that appear mathematically valid but are contextually implausible.
2. **Triangulation Methodology**:
   - **Domain Reference Ranges**: Compare reported prices against verified regional crop economic ranges (`crop_price_ranges`).
   - **Threshold Proximity**: Isolate extreme scale values ($> ₹20,000$).
   - **Chronological Neighbor Comparison**: Contrast suspect values with previous-day ($t-1$) and next-day ($t+1$) quotes within the same `(market, commodity)` series.
   - **Outlier Flag Cross-Referencing**: Correlate entries with the statistical IQR flags computed in Problem 13.
3. **Preservation Principle**: Never use blunt global operations (like `.clip()`). Detection and multi-attribute inspection prepare the dataset for context-aware time-series smoothing and imputation.

---

## Sub-section 14.1: Domain Price Range Baseline & Anomaly Profiling

#### Step 1: Define Expected Benchmark Price Ranges per Commodity

In [ ]:
# ============================================================
# PROBLEM 14: INCORRECT DATA ENTRY
# STEP 1: CHECK SUSPICIOUS PRICE VALUES
# ============================================================

crop_price_ranges = {
    "Cotton": (3000, 8000),
    "Rice": (4000, 7000),
    "Wheat": (2000, 4000),
    "Soyabean": (3500, 8000),
    "Pea": (3500, 6000),
    "Orange": (7000, 15000),
    "Potato": (3000, 6000),
    "Onion": (3000, 7000),
    "Tomato": (2000, 8000),
    "Green Chilli": (2000, 8000),
    "Watermelon": (1000, 4000),
    "Mango": (4000, 10000)
}

#### Step 2: Detect Price Values Deviating from Expected Domain Ranges

In [ ]:
# ============================================================
# STEP 2: DETECT PRICE VALUES OUTSIDE EXPECTED RANGE
# ============================================================

for col in ["min_price", "modal_price", "max_price"]:

    suspicious = pd.Series(False, index=df.index)

    for crop, (low, high) in crop_price_ranges.items():

        mask = df["commodity"] == crop

        suspicious |= (
            mask &
            (
                (df[col] < low) |
                (df[col] > high)
            )
        )

    print(
        f"{col}: {suspicious.sum()} suspicious values"
    )

    display(
        df.loc[
            suspicious,
            ["date", "market", "commodity", col]
        ].head(20)
    )

## Sub-section 14.2: Magnitude Shifts & Temporal Neighbor Verification

#### Step 3: Identify Severe Order-of-Magnitude Anomalies ($> ₹20,000$)

In [ ]:
# ============================================================
# STEP 3: FIND EXTREMELY HIGH PRICE VALUES
# ============================================================

for col in ["min_price", "modal_price", "max_price"]:

    suspicious = df[col] > 20000

    print(f"\n{col} values greater than ₹20,000:")
    
    display(
        df.loc[
            suspicious,
            ["date", "market", "commodity", col]
        ].head(20)
    )

#### Step 4: Compare Suspicious Entries Against Chronological Neighbor Records

In [ ]:
# ============================================================
# STEP 4: CHECK NEIGHBORING RECORDS
# ============================================================

df_sorted = df.sort_values(
    ["commodity", "market", "date"]
).copy()

for crop in ["Rice", "Wheat", "Cotton"]:

    example = df_sorted[
        df_sorted["commodity"] == crop
    ]

    print(f"\nRecent records for {crop}:")

    display(
        example[
            [
                "date",
                "market",
                "commodity",
                "min_price",
                "modal_price",
                "max_price"
            ]
        ].head(20)
    )

## Sub-section 14.3: Cross-Referencing Outlier Flags & Strategic Policy

#### Step 5: Cross-Reference Potential Data-Entry Errors With Statistical Outlier Flags

In [ ]:
# ============================================================
# STEP 5: DISPLAY POTENTIAL INCORRECT ENTRIES
# ============================================================

suspicious_entries = df[
    df["any_outlier"] == True
]

display(
    suspicious_entries[
        [
            "date",
            "market",
            "commodity",
            "min_price",
            "modal_price",
            "max_price",
            "any_outlier"
        ]
    ].head(30)
)

#### Step 6: Methodological Principles for Handling Data Entry Errors

> **Methodological Principle:**  
> Avoid naive transformations like `df.clip()`. Confirmed data entry anomalies should be converted to `NaN` and reconstructed using grouped time-series linear interpolation and local median smoothing while strictly enforcing price hierarchy (`min_price <= modal_price <= max_price`).

---
# SECTION 15: Problem 15 – Post-Sanitization Imputation of Secondary Missing Values

### Theoretical & Domain Architecture:
1. **Secondary Missing Value Genesis**: While baseline missing values were handled in Section 1, our sanitization steps (Problem 11 & 12) converted domain-invalid and physically impossible observations into `np.nan` to preserve dataset row count integrity.
2. **Stratified Hierarchical Imputation Protocols**:
   - **Weather (`temperature`, `humidity`)**: Impute via **`(date, market)` median** $\rightarrow$ **`market` median** $\rightarrow$ **global median** (since atmospheric conditions are identical across all commodities in the same market on the same date).
   - **Volumes (`arrival_quantity`, `import_quantity`, `export_quantity`)**: Impute via **`(commodity, market)` grouped linear interpolation** $\rightarrow$ **`(commodity, market)` median** $\rightarrow$ **global median**.
   - **Prices (`min_price`, `modal_price`, `max_price`)**: Impute via **`(commodity, market)` grouped linear interpolation** $\rightarrow$ **`(commodity, market)` median** $\rightarrow$ **global median**.
   - **Policy Flag (`msp_applicable`)**: Deterministically reconstruct via government agricultural MSP policy schedule (Crops eligible for statutory MSP: `Cotton`, `Rice`, `Soyabean`, `Wheat` $\rightarrow 1$, all others $\rightarrow 0$).
3. **Integrity & Zero Leakage**: Strictly avoid filling identifiers/categorical columns with arbitrary statistics, and ensure zero remaining missing cells dataset-wide.

---

## Sub-section 15.1: Missing Values Audit Following Sanitization

#### Step 1 & 2: Audit Columns Containing Null Values

In [ ]:
# ============================================================
# PROBLEM 15: REMAINING MISSING VALUES
# STEP 1 & 2: AUDIT MISSING VALUES
# ============================================================

print("Missing values in each column:")

missing_counts = df.isnull().sum()
print(missing_counts[missing_counts > 0])

missing_columns = (
    df.isnull()
    .sum()
    .loc[lambda x: x > 0]
)

print("\nColumns having missing values:")
print(missing_columns)

## Sub-section 15.2: Weather Features Hierarchical Imputation

#### Step 3: Impute Temperature & Humidity via Date + Market Medians

In [ ]:
# ============================================================
# TEMPERATURE AND HUMIDITY
# ============================================================

for col in ["temperature", "humidity"]:

    df[col] = (
        df.groupby(["date", "market"])[col]
        .transform(lambda x: x.fillna(x.median()))
    )

    # Fallback: market median
    df[col] = (
        df.groupby("market")[col]
        .transform(lambda x: x.fillna(x.median()))
    )

    # Final fallback: overall median
    df[col] = df[col].fillna(df[col].median())

## Sub-section 15.3: Supply & Trade Volume Features Imputation

#### Step 4: Impute Arrival & Trade Quantities via Grouped Linear Interpolation

In [ ]:
# ============================================================
# QUANTITY COLUMNS
# ============================================================

quantity_columns = [
    "arrival_quantity",
    "import_quantity",
    "export_quantity"
]

for col in quantity_columns:

    # Interpolation within crop + market
    df[col] = (
        df.groupby(["commodity", "market"])[col]
        .transform(
            lambda x: x.interpolate(
                method="linear",
                limit_direction="both"
            )
        )
    )

    # Fallback: crop + market median
    df[col] = (
        df.groupby(["commodity", "market"])[col]
        .transform(lambda x: x.fillna(x.median()))
    )

    # Final fallback
    df[col] = df[col].fillna(df[col].median())

## Sub-section 15.4: Price Features Imputation

#### Step 5: Impute Min, Modal, and Max Prices via Grouped Linear Interpolation

In [ ]:
# ============================================================
# PRICE COLUMNS
# ============================================================

price_columns = [
    "min_price",
    "modal_price",
    "max_price"
]

for col in price_columns:

    df[col] = (
        df.groupby(["commodity", "market"])[col]
        .transform(
            lambda x: x.interpolate(
                method="linear",
                limit_direction="both"
            )
        )
    )

    # Fallback: crop + market median
    df[col] = (
        df.groupby(["commodity", "market"])[col]
        .transform(lambda x: x.fillna(x.median()))
    )

    # Final fallback: overall median
    df[col] = df[col].fillna(df[col].median())

## Sub-section 15.5: Policy Indicator Deterministic Reconstruction

#### Step 6: Derive `msp_applicable` From Crop Policy Eligibility

In [ ]:
# ============================================================
# MSP APPLICABLE
# ============================================================

msp_crops = [
    "Cotton",
    "Rice",
    "Soyabean",
    "Wheat"
]

df.loc[
    df["commodity"].isin(msp_crops),
    "msp_applicable"
] = 1

df.loc[
    ~df["commodity"].isin(msp_crops),
    "msp_applicable"
] = 0

## Sub-section 15.6: Final Missing Values Verification

#### Step 7 & 8: Verify Complete Dataset Completeness (0 Missing Cells)

In [ ]:
# ============================================================
# FINAL MISSING VALUE CHECK
# ============================================================

print("Remaining missing values:")

remaining_missing = df.isnull().sum()
print(remaining_missing[remaining_missing > 0])

print(
    "\nTotal missing cells across dataset:",
    df.isnull().sum().sum()
)

---
# SECTION 16: Problem 16 – Price Hierarchy Consistency Validation (min $\le$ modal $\le$ max)

### Economic & Domain Invariant:
In agricultural APMC mandis, price quotes for any given commodity must strictly satisfy the pricing hierarchy:
$$\text{min\_price} \le \text{modal\_price} \le \text{max\_price}$$
Violations (e.g., $\text{min\_price} > \text{modal\_price}$ or $\text{modal\_price} > \text{max\_price}$) represent transposition/swapping errors occurring during manual or automated transcription.

### Corrective Architecture:
* **Detection**: Isolate and enumerate all multi-attribute pricing order violations.
* **Disaggregated Component Audit**: Individually quantify each specific sub-condition violation.
* **Row-Wise Vectorized Sorting**: Reconstruct the natural monotonic hierarchy using NumPy row-wise array sorting (`price_values.sort(axis=1)`).
* **Post-Correction Verification**: Validate 0 remaining pricing order contradictions.

---

## Sub-section 16.1: Price Hierarchy Anomaly Discovery & Inspection

#### Step 1: Detect Rows With Inverted Pricing Hierarchy

In [ ]:
# ============================================================
# PROBLEM 16: PRICE RELATIONSHIP VALIDATION
# STEP 1: DETECT INCORRECT PRICE RELATIONSHIPS
# ============================================================

invalid_price_rows = df[
    (df["min_price"] > df["modal_price"]) |
    (df["modal_price"] > df["max_price"]) |
    (df["min_price"] > df["max_price"])
]

print(
    "Number of rows with incorrect price relationship:",
    len(invalid_price_rows)
)

display(
    invalid_price_rows[
        [
            "date",
            "market",
            "commodity",
            "min_price",
            "modal_price",
            "max_price"
        ]
    ].head(20)
)

#### Step 2: Quantify Separate Sub-Condition Inversion Counts

In [ ]:
# ============================================================
# STEP 2: CHECK SUB-CONDITIONS SEPARATELY
# ============================================================

print(
    "min_price > modal_price:",
    (df["min_price"] > df["modal_price"]).sum()
)

print(
    "modal_price > max_price:",
    (df["modal_price"] > df["max_price"]).sum()
)

print(
    "min_price > max_price:",
    (df["min_price"] > df["max_price"]).sum()
)

## Sub-section 16.2: Vectorized Price Hierarchy Restoration & Verification

#### Step 3: Correct Price Ordering via Vectorized Row-Wise Sorting

> **Domain Rationale:** Scrambled price columns resulting from transcription swaps within the same market transaction are restored to their true physical order: $\min \le \text{modal} \le \max$.

In [ ]:
# ============================================================
# STEP 3: CORRECT PRICE ORDER
# ============================================================

price_values = df[
    ["min_price", "modal_price", "max_price"]
].values

price_values.sort(axis=1)

df[
    ["min_price", "modal_price", "max_price"]
] = price_values

#### Step 4: Final Verification of Price Ordering Invariant

In [ ]:
# ============================================================
# STEP 4: FINAL VALIDATION
# ============================================================

invalid_price_rows = df[
    (df["min_price"] > df["modal_price"]) |
    (df["modal_price"] > df["max_price"]) |
    (df["min_price"] > df["max_price"])
]

print(
    "Remaining incorrect price relationships:",
    len(invalid_price_rows)
)

---
# SECTION 17: Problem 17 – Minimum Support Price (MSP) Policy Consistency Validation

### Policy & Domain Invariant Framework:
1. **Statutory Coverage**: Under national agricultural pricing policy, statutory MSP regimes apply strictly to designated staples and commercial cash crops. In our portfolio:
   - **MSP Applicable ($=1$)**: `Cotton`, `Rice`, `Soyabean`, `Wheat`
   - **Non-MSP Crops ($=0$)**: `Pea`, `Orange`, `Potato`, `Onion`, `Tomato`, `Green Chilli`, `Watermelon`, `Mango`
2. **Consistency Invariant**: `msp_applicable` must deterministically match commodity eligibility, and for all rows where `msp_applicable == 1`, the numeric quote `msp` must be strictly positive ($>0$) and non-null.

---

## Sub-section 17.1: MSP Flag Inconsistency Discovery

#### Step 1: Check Recorded vs Policy Expected Applicability

In [ ]:
# ============================================================
# PROBLEM 17: MSP VALIDATION
# STEP 1: CHECK MSP APPLICABILITY
# ============================================================

msp_crops = [
    "Cotton",
    "Rice",
    "Soyabean",
    "Wheat"
]

expected_msp_applicable = df["commodity"].isin(
    msp_crops
).astype(int)

invalid_msp_applicable = df[
    df["msp_applicable"] != expected_msp_applicable
]

print(
    "Rows with incorrect msp_applicable:",
    len(invalid_msp_applicable)
)

display(
    invalid_msp_applicable[
        [
            "commodity",
            "msp",
            "msp_applicable"
        ]
    ].head(20)
)

## Sub-section 17.2: Deterministic Policy Enforcement

#### Step 2: Enforce Canonical MSP Applicability Based on Commodity Eligibility

In [ ]:
# ============================================================
# STEP 2: CORRECT MSP APPLICABILITY
# ============================================================

df["msp_applicable"] = (
    df["commodity"]
    .isin(msp_crops)
    .astype(int)
)

## Sub-section 17.3: Numerical Support Price Validation

#### Step 3: Validate Positivity and Completeness of MSP Quotes for Covered Crops

In [ ]:
# ============================================================
# STEP 3: CHECK MSP VALUES
# ============================================================

invalid_msp = df[
    (df["msp_applicable"] == 1) &
    (
        df["msp"].isna() |
        (df["msp"] <= 0)
    )
]

print(
    "Rows with missing/invalid MSP:",
    len(invalid_msp)
)

display(
    invalid_msp[
        [
            "date",
            "market",
            "commodity",
            "msp",
            "msp_applicable"
        ]
    ].head(20)
)

#### Step 4: Audit Grouped MSP Applicability Across All 12 Commodities

In [ ]:
print("MSP applicability by crop:")
print(
    df.groupby("commodity")["msp_applicable"]
      .unique()
)

## Sub-section 17.4: Final Policy & Numerical Verification

#### Step 5: Final Comprehensive MSP Validation (0 Discrepancies)

In [ ]:
# ============================================================
# STEP 5: FINAL MSP VALIDATION
# ============================================================

expected_msp_applicable = (
    df["commodity"]
    .isin(msp_crops)
    .astype(int)
)

print(
    "Incorrect MSP applicability:",
    (
        df["msp_applicable"] != expected_msp_applicable
    ).sum()
)

print(
    "Invalid MSP values for applicable crops:",
    (
        (df["msp_applicable"] == 1) &
        (
            df["msp"].isna() |
            (df["msp"] <= 0)
        )
    ).sum()
)

---
# SECTION 18: Problem 18 – Business-Key Semantic Duplicate Validation (`date + market + commodity`)

### Theoretical & Domain Architecture:
1. **Exact Duplicates vs Business-Key Duplicates**:
   - **Exact Duplicates (Problem 2)**: Records where every single column is 100% identical.
   - **Business-Key Duplicates (Problem 18)**: Records sharing the same composite domain primary key `[date, market, commodity]`, even if minor noisy attributes differ.
2. **Grid Completeness Proof**:
   In a structured daily panel dataset, there exists exactly one observation per market per commodity per calendar date:
   $$\text{Expected Rows} = |\text{Dates}| \times |\text{Markets}| \times |\text{Commodities}| = 1,734 \times 13 \times 12 = 270,504$$

---

## Sub-section 18.1: Business-Key Duplicate Detection & Exploration

#### Step 1: Detect Duplicate Combinations of Composite Key (`date`, `market`, `commodity`)

In [ ]:
# ============================================================
# PROBLEM 18: BUSINESS-KEY DUPLICATES
# STEP 1: CHECK DUPLICATE DATE + MARKET + COMMODITY
# ============================================================

business_key = [
    "date",
    "market",
    "commodity"
]

duplicate_key_count = df.duplicated(
    subset=business_key,
    keep=False
).sum()

print(
    "Rows involved in business-key duplicates:",
    duplicate_key_count
)

#### Step 2: Display Business-Key Duplicate Records (Sorted Group View)

In [ ]:
duplicate_key_rows = df[
    df.duplicated(
        subset=business_key,
        keep=False
    )
].sort_values(business_key)

print("Duplicate business-key records:")

display(
    duplicate_key_rows[
        [
            "date",
            "market",
            "commodity",
            "arrival_quantity",
            "min_price",
            "modal_price",
            "max_price"
        ]
    ].head(30)
)

#### Step 3: Count and Group Business-Key Redundancy Frequencies

In [ ]:
duplicate_groups = (
    df.groupby(
        business_key,
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

duplicate_groups = duplicate_groups[
    duplicate_groups["count"] > 1
]

print(
    "Number of duplicate business-key groups:",
    len(duplicate_groups)
)

display(duplicate_groups.head(20))

## Sub-section 18.2: Deduplication & Grid Integrity Verification

#### Step 4: Remove Business-Key Duplicates (Retain First Chronological Occurrence)

In [ ]:
# ============================================================
# STEP 4: REMOVE BUSINESS-KEY DUPLICATES
# ============================================================

before_rows = len(df)

df = (
    df.sort_values(
        ["date", "market", "commodity"]
    )
    .drop_duplicates(
        subset=business_key,
        keep="first"
    )
    .reset_index(drop=True)
)

after_rows = len(df)

print("Rows before:", before_rows)
print("Rows after :", after_rows)
print("Rows removed:", before_rows - after_rows)

#### Step 5 & 6: Final Verification of Zero Business-Key Duplicates & Panel Dimension Match

In [ ]:
# ============================================================
# STEP 5 & 6: FINAL VERIFICATION
# ============================================================

remaining_duplicates = df.duplicated(
    subset=business_key
).sum()

print(
    "Remaining business-key duplicates:",
    remaining_duplicates
)

expected_rows = (
    df["date"].nunique()
    * df["market"].nunique()
    * df["commodity"].nunique()
)

print("\nExpected rows (Dates × Markets × Commodities):", expected_rows)
print("Actual rows in dataset                       :", len(df))

---
# SECTION 19: Problem 19 – Comprehensive Final Quality Assurance, Integrity Audit & Export

### Quality Assurance Master Checklist:
- [x] **Dimensions & Structure**: Verify row count ($270,504$) and column schema.
- [x] **Temporary Column Cleanup**: Remove analysis helpers (`month`, `*_outlier`, `any_outlier`).
- [x] **Completeness**: $0$ missing cells across all features.
- [x] **Duplicate Immunity**: $0$ exact duplicates and $0$ business-key semantic duplicates (`[date, market, commodity]`).
- [x] **Geographical Scope**: Strictly $1$ state (`Maharashtra`), $1$ district (`Nagpur`), $13$ markets, and $12$ commodities.
- [x] **Economic Coherence**: $\text{min\_price} \le \text{modal\_price} \le \text{max\_price}$ for $100\%$ of records.
- [x] **Domain Constraints**: Non-negative supply volumes/rates and bounded weather metrics.
- [x] **Schema Integrity**: Canonical datatypes.
- [x] **Golden File Export**: Save standardized final dataset as `nagpur_crop_price_CLEANED.csv`.

---

## Sub-section 19.1: Dataset Dimensions & Helper Column Cleanup

#### Step 1: Check Current Dataset Dimensions

In [ ]:
# ============================================================
# FINAL VALIDATION
# STEP 1: DATASET SHAPE
# ============================================================

print("Number of rows    :", df.shape[0])
print("Number of columns :", df.shape[1])

#### Step 2: Remove Temporary Cleaning & Analysis Columns

In [ ]:
# ============================================================
# STEP 2: REMOVE TEMPORARY COLUMNS
# ============================================================

temporary_columns = [
    "month",
    "arrival_quantity_outlier",
    "min_price_outlier",
    "modal_price_outlier",
    "max_price_outlier",
    "any_outlier"
]

df.drop(
    columns=[
        col for col in temporary_columns
        if col in df.columns
    ],
    inplace=True
)

## Sub-section 19.2: Rigorous Quality Assurance Invariant Audits

#### Step 3: Verify Zero Missing Cells

In [ ]:
print("Total missing cells:",
      df.isnull().sum().sum())

print("\nMissing values by column:")
print(df.isnull().sum())

#### Step 4 & 5: Verify Zero Exact & Business-Key Duplicates

In [ ]:
print(
    "Exact duplicate rows:",
    df.duplicated().sum()
)

business_key = [
    "date",
    "market",
    "commodity"
]

print(
    "Business-key duplicates:",
    df.duplicated(
        subset=business_key
    ).sum()
)

#### Step 6: Verify Geographical & Entity Uniformity

In [ ]:
print("States:")
print(df["state"].unique())

print("\nDistricts:")
print(df["district"].unique())

print("\nNumber of markets:",
      df["market"].nunique())

print("\nNumber of crops:",
      df["commodity"].nunique())

#### Step 7: Verify Price Hierarchy Constraint Invariant

In [ ]:
price_errors = df[
    (df["min_price"] > df["modal_price"]) |
    (df["modal_price"] > df["max_price"])
]

print(
    "Price relationship errors:",
    len(price_errors)
)

#### Step 8: Verify Domain Constraints (Non-Negative Values & Valid Ranges)

In [ ]:
print(
    "Negative arrival quantity:",
    (df["arrival_quantity"] < 0).sum()
)

print(
    "Negative import quantity:",
    (df["import_quantity"] < 0).sum()
)

print(
    "Negative export quantity:",
    (df["export_quantity"] < 0).sum()
)

print(
    "Negative fuel cost:",
    (df["fuel_cost"] < 0).sum()
)

print(
    "Invalid humidity:",
    ((df["humidity"] < 0) |
     (df["humidity"] > 100)).sum()
)

print(
    "Invalid MSP applicable:",
    (~df["msp_applicable"].isin([0, 1])).sum()
)

#### Step 9 & 10: Final Schema Audit & Dataset Executive Summary

In [ ]:
print("\nFinal data types:")
print(df.dtypes)

print("\n" + "=" * 50)
print("FINAL DATASET EXECUTIVE SUMMARY")
print("=" * 50)

print("Rows       :", df.shape[0])
print("Columns    :", df.shape[1])
print("Duplicates :", df.duplicated().sum())
print("Missing    :", df.isnull().sum().sum())

print("\nDate range:")
print("From:", df["date"].min())
print("To  :", df["date"].max())

print("\nMarkets:", df["market"].nunique())
print("Crops  :", df["commodity"].nunique())

## Sub-section 19.3: Golden File Export

#### Step 11: Save Cleaned Dataset to CSV

In [ ]:
# ============================================================
# SAVE CLEAN DATASET
# ============================================================

output_file = "nagpur_crop_price_CLEANED.csv"

df.to_csv(
    output_file,
    index=False
)

print(
    "Cleaned dataset saved successfully as:",
    output_file
)